In [2]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

In [78]:
class CustomDataset(Dataset):
  def __init__(self,filepath):
    self.filepath = filepath
    self.data = pd.read_csv(filepath)


    self.x_mean = self.data['X'].mean()
    self.x_std = self.data['X'].std()

  def store_features(self):
    x_features = self.data['X']
    y_labels = self.data['y']

  def __getitem__(self, index):
   row = self.data.iloc[index]
   sample = (row["X"] - self.x_mean) / self.x_std
   label = float(row["y"])
   return sample, label

  def __len__(self):
    return len(self.data)

  def support_dataloader(self,batch_size=32,shuffle=False):
    dataloader = DataLoader(self,batch_size,shuffle)
    return dataloader

In [79]:
num_samples = 50

X = torch.randn(num_samples, 1) * 500
y = torch.randint(0, 2, (num_samples,)).unsqueeze(1)
X,y

(tensor([[ 1.3893e+02],
         [ 3.6710e+02],
         [-1.8678e+02],
         [-1.9760e+02],
         [-6.2243e+02],
         [-2.1298e+02],
         [-4.6305e+02],
         [ 1.6743e+02],
         [ 5.1243e+02],
         [-3.4569e+02],
         [-2.6625e+02],
         [-1.0031e+03],
         [ 2.6875e+02],
         [ 2.1235e+02],
         [-1.9910e+02],
         [ 2.1525e+02],
         [-3.4820e+01],
         [-9.5934e+02],
         [-5.9293e+02],
         [ 6.9799e+02],
         [ 1.5895e+02],
         [ 5.2426e+02],
         [-4.4448e+02],
         [-7.4742e+01],
         [ 5.7607e+00],
         [-1.8244e+02],
         [-7.2816e+01],
         [ 5.7078e+02],
         [ 1.0096e+03],
         [-5.9941e+02],
         [-4.4847e+01],
         [ 5.0676e+02],
         [-2.5935e+02],
         [ 4.0045e+02],
         [-2.5727e+01],
         [ 1.2636e+02],
         [ 3.9752e+02],
         [ 6.3209e+02],
         [-2.0198e+01],
         [-9.5622e+01],
         [ 3.0863e+02],
         [-7.330

In [80]:
X.shape, y.shape

(torch.Size([50, 1]), torch.Size([50, 1]))

In [81]:
data = torch.cat((X, y), dim=1)
df = pd.DataFrame(data.numpy(), columns=['X', 'y'])
df.to_csv('data.csv', index=False)

In [82]:
# @title

from sklearn.model_selection import train_test_split
data = pd.read_csv('data.csv')
x_train,x_test,y_train,y_test = train_test_split(data['X'],data['y'],test_size=0.2,random_state=42)

In [87]:
from torch.utils.data import random_split
train_size = int(0.8 * len(data))
test_size = len(data) - train_size


train_dataset, test_dataset = random_split(
    CD1, [train_size, test_size]
)

In [86]:
CD1 = CustomDataset('data.csv')

In [85]:
CD1.__getitem__(2)

(np.float64(-0.2237463031905498), 1.0)

In [65]:
CD1.__len__()

50

In [88]:
train_loader = DataLoader(train_dataset,batch_size=48,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=48,shuffle=True)

In [92]:
import torch.nn as nn
import torch.optim as optim
class LinearRegressionModel(nn.Module):
    def __init__(self):
        super(LinearRegressionModel, self).__init__()
        self.linear = nn.Linear(1, 1)

    def forward(self, x):
        return self.linear(x)


model = LinearRegressionModel()
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.001)

In [93]:
torch.manual_seed(42)

epochs = 2000
for e in range(epochs):
  model.train()
  for batch_X, batch_y in train_loader:

        batch_X = batch_X.float().unsqueeze(1)
        batch_y = batch_y.float().unsqueeze(1)

        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

  model.eval()
  with torch.no_grad():
    for batch_X, batch_y in test_loader:

        batch_X = batch_X.float().unsqueeze(1)
        batch_y = batch_y.float().unsqueeze(1)

        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)

  if (e + 1) % 100 == 0:
      print(f"Epoch [{e + 1}/{epochs}], Loss: {loss.item():.4f}")

Epoch [100/2000], Loss: 0.2657
Epoch [200/2000], Loss: 0.2582
Epoch [300/2000], Loss: 0.2552
Epoch [400/2000], Loss: 0.2549
Epoch [500/2000], Loss: 0.2560
Epoch [600/2000], Loss: 0.2578
Epoch [700/2000], Loss: 0.2599
Epoch [800/2000], Loss: 0.2620
Epoch [900/2000], Loss: 0.2640
Epoch [1000/2000], Loss: 0.2658
Epoch [1100/2000], Loss: 0.2673
Epoch [1200/2000], Loss: 0.2687
Epoch [1300/2000], Loss: 0.2699
Epoch [1400/2000], Loss: 0.2708
Epoch [1500/2000], Loss: 0.2716
Epoch [1600/2000], Loss: 0.2723
Epoch [1700/2000], Loss: 0.2729
Epoch [1800/2000], Loss: 0.2734
Epoch [1900/2000], Loss: 0.2737
Epoch [2000/2000], Loss: 0.2741
